In [ ]:
# Set environment variables to suppress TensorFlow logs
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress all logs except FATAL
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'   # Disable oneDNN optimizations

# Consolidated Imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from tensorflow.keras import mixed_precision
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Step 1: Import mixed_precision before other TensorFlow Keras modules
mixed_precision.set_global_policy('float32')

# Verify the policy is set correctly
print("Mixed Precision Policy:", mixed_precision.global_policy())

# Optionally, set TensorFlow logger level to ERROR
tf.get_logger().setLevel('ERROR')
# Suppress TensorFlow logs
# Move the environment variable setting before importing TensorFlow

# Enable XLA
# tf.config.optimizer.set_jit(True)  # Disable XLA


In [ ]:
import os  # Import os module
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# List of categorical columns
categorical_columns = [
    'Occupation', 'Customer Feedback', 'Education Level', 'Location',
    'Policy Type', 'Smoking Status', 'Exercise Frequency', 'Property Type',
    'Gender', 'Marital Status'
]

# Define cleaned categorical columns for later use
cleaned_categorical_columns = [
    'Occupation', 'CustomerFeedback', 'EducationLevel', 'Location',
    'PolicyType', 'SmokingStatus', 'ExerciseFrequency', 'PropertyType',
    'Gender', 'MaritalStatus'
]

# Add a preprocessing function
def preprocess_data(file, cleaned_categorical_columns):
    import numpy as np
    import pandas as pd
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_selection import SelectKBest, f_regression
    import joblib

    # Load the dataset
    data = pd.read_csv(f'data/{file}.csv')
    print(f"Raw data loaded from 'data/{file}.csv'")

    # Handle Missing Values
    data[categorical_columns] = data[categorical_columns].fillna('Unknown')
    numeric_columns = data.select_dtypes(include=[np.number]).columns
    data[numeric_columns] = data[numeric_columns].fillna(data[numeric_columns].mean())

    # Process Dates
    if 'Policy Start Date' in data.columns:
        data['Policy Start Date'] = pd.to_datetime(data['Policy Start Date'], errors='coerce')
        data['Signup Month'] = data['Policy Start Date'].dt.month
        data = data.drop('Policy Start Date', axis=1)

    # Clean Column Names
    data.columns = data.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)
    cleaned_categorical_columns = [col.replace(' ', '') for col in cleaned_categorical_columns]

    # One-Hot Encoding
    data = pd.get_dummies(data, columns=cleaned_categorical_columns, drop_first=True)

    # Convert all remaining columns to numeric
    for col in data.columns:
        if col != 'id':  # Preserve id column
            data[col] = pd.to_numeric(data[col], errors='coerce')

    # Feature Engineering: Interaction Terms
    variables = [
        'Age', 'Gender', 'AnnualIncome', 'EducationLevel', 'Occupation',
        'HealthScore', 'PolicyType', 'PreviousClaims', 'CreditScore',
        'SmokingStatus', 'ExerciseFrequency'
    ]

    numerical_columns = [col for col in variables if col not in cleaned_categorical_columns]

    interaction_list = []
    for num_col in numerical_columns:
        num_series = data[num_col]
        categorical_cols = [col for col in data.columns if any(col.startswith(f'{cat}_') for cat in cleaned_categorical_columns)]
        interaction_terms = data[categorical_cols].multiply(num_series, axis=0)
        interaction_terms = interaction_terms.add_prefix(f'{num_col}_')
        interaction_list.append(interaction_terms)
    interaction_df = pd.concat(interaction_list, axis=1)

    # Final Cleanup
    data = data.fillna(0)
    data.replace([np.inf, -np.inf], 0, inplace=True)

    # Combine with Interaction Features
    data = pd.concat([data, interaction_df], axis=1).copy()

    # Save cleaned data
    data.to_csv(f'cleaned_{file}.csv', index=False)
    print(f"Cleaned {file} saved to 'cleaned_{file}.csv'")
    print("Columns after cleaning:", data.columns.tolist())

    # Update feature names using X instead of X_encoded
    if file == 'train':
        # Define features and target
        X = data.drop(['id', 'PremiumAmount'], axis=1)
        y = data['PremiumAmount']

        # Remove duplicated saving of feature_names.joblib
        # joblib.dump(feature_names, 'feature_names.joblib')
        # print("Feature names saved.")

        # Save feature names once
        feature_names = X.columns.tolist()
        joblib.dump(cleaned_categorical_columns, 'cleaned_categorical_columns.joblib')
        print("Categorical columns saved.")

        # Save feature names
        feature_names = X.columns.tolist()
        joblib.dump(feature_names, 'feature_names.joblib')
        print("Feature names saved.")

        # Scale the data
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        joblib.dump(scaler, 'scaler.joblib')
        print("Scaler saved.")

        # Feature selection
        selector = SelectKBest(f_regression, k=20)
        X_selected = selector.fit_transform(X_scaled, y)
        joblib.dump(selector, 'selector.joblib')
        print("Selector saved.")

        # Save selected feature names
        selected_features = X.columns[selector.get_support()].tolist()
        joblib.dump(selected_features, 'selected_features.joblib')
        print("Selected feature names saved.")

    return

# Update existing preprocessing cells to use the new function
for file in ['train', 'test']:
    preprocess_data(file, cleaned_categorical_columns)

# Add a check to confirm preprocessing completed for 'train'
print("Preprocessing completed for both train and test datasets.")


In [ ]:
import numpy as np
import pandas as pd
from keras_tuner import HyperModel, RandomSearch, BayesianOptimization
import traceback
import logging
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError  # Import the loss class
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, TensorBoard
from tensorflow.keras.metrics import MeanSquaredError  # Import the metric class
from tensorflow.keras.layers import BatchNormalization
from sklearn.model_selection import KFold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.regularizers import l2
import joblib
import os  # Add this import at the beginning

# Setup logging
logging.basicConfig(filename='training.log', level=logging.INFO, 
                    format='%(asctime)s:%(levelname)s:%(message)s')

# Load cleaned train data
data = pd.read_csv('cleaned_train.csv')
print("Cleaned data loaded from 'cleaned_train.csv'")
print("Available columns:", data.columns.tolist())

# Define features and target
X = data.drop(['id', 'PremiumAmount'], axis=1)  # Drop both id and target
y = data['PremiumAmount']

# Handle missing values
X.fillna(0, inplace=True)

cleaned_categorical_columns = joblib.load('cleaned_categorical_columns.joblib')
print("Categorical columns loaded.")

feature_names = joblib.load('feature_names.joblib')
print("Feature names loaded.")

scaler = joblib.load('scaler.joblib')
print("Scaler loaded.")

selector = joblib.load('selector.joblib')
print("Selector loaded.")

# Update feature names using X instead of X_encoded
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'feature_names.joblib')
print("Feature names saved.")

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, 'scaler.joblib')
print("Scaler saved.")

# Feature selection
selector = SelectKBest(f_regression, k=20)
X_selected = selector.fit_transform(X_scaled, y)
joblib.dump(selector, 'selector.joblib')
print("Selector saved.")

# Save selected feature names
selected_features = X.columns[selector.get_support()].tolist()
joblib.dump(selected_features, 'selected_features.joblib')
print("Selected feature names saved.")

# Convert target variable
y = np.log1p(y)

# Initialize K-Fold Cross-Validation
num_folds = 10  # Increased from 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)  # Each fold will initialize a new model

# Modify the create_dataset function to ensure proper data batching
def create_dataset(X, y, batch_size=64):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# Initialize lists to store results and models
all_losses = []
all_maes = []
best_models = []

# Define the number of folds
num_folds = 5

def train_fold(fold_data):
    fold, train_index, val_index = fold_data
    model_path = f'insurance_best_model_fold_{fold}.h5'

    # Check if the model for this fold already exists
    if os.path.exists(model_path):
        print(f"Model for fold {fold} already exists. Skipping.")
        return {
            'fold': fold,
            'loss': None,
            'mae': None,
            'best_model': None
        }

    # Prepare the data for this fold
    X_train_fold, X_val_fold = X_selected[train_index], X_selected[val_index]
    y_train_fold, y_val_fold = y[train_index], y[val_index]

    # Define the hypermodel with adjusted learning rate range
    class InsuranceHyperModel(HyperModel):
        def __init__(self, input_shape):
            super().__init__()  # Call parent class's __init__
            self.input_shape = input_shape

        def build(self, hp):
            model = Sequential()
            model.add(Input(shape=(self.input_shape,)))
            for i in range(hp.Int('layers', 2, 4)):  # Limit layers based on best models
                model.add(Dense(
                    units=hp.Int(f'units_{i}', 512, 2048, step=128),  # Focus on higher units
                    activation='relu',
                    kernel_regularizer=l2(0.001)
                ))
                model.add(BatchNormalization())
                model.add(Dropout(
                    rate=hp.Float(f'dropout_{i}', 0.0, 0.2, step=0.1)  # Reduce dropout rates
                ))
            model.add(Dense(1, activation='linear'))
            model.compile(
                optimizer=Adam(
                    learning_rate=hp.Float('lr', 1e-4, 2e-3, sampling='log')  # Narrowed learning rate range
                ),
                loss='mean_squared_error',
                metrics=['mae']
            )
            return model

    hypermodel = InsuranceHyperModel(input_shape=X_train_fold.shape[1])

    # proceed with the new plan
    tuner = BayesianOptimization(
        hypermodel,
        objective='val_mae',
        max_trials=10,  # New plan with reduced trials
        directory=f'tuner_fold_{fold}',
        project_name='insurance',
        overwrite=False
    )

    # Early stopping callback
    early_stopping = EarlyStopping(
        monitor='val_mae',
        patience=2,
        restore_best_weights=True
    )

    # Set up TensorBoard log directory
    log_dir = f'logs/fold_{fold}'
    tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

    # Hyperparameter search with limited epochs
    tuner.search(
        X_train_fold, y_train_fold,
        epochs=10,  # Train each model for 10 epochs during tuning
        validation_data=(X_val_fold, y_val_fold),
        callbacks=[early_stopping, tensorboard_callback],
        verbose=1
    )

    # Get the best hyperparameters
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

    # Build the model with the best hyperparameters
    model = hypermodel.build(best_hps)

    # Training loop with incremental epochs
    epochs_per_increment = 10
    total_epochs_trained = 10  # Already trained for 10 epochs during tuning
    best_val_mae = float('inf')
    no_improvement_counter = 0

    while no_improvement_counter < 2:
        # Train the model for an increment of epochs
        history = model.fit(
            X_train_fold, y_train_fold,
            initial_epoch=total_epochs_trained,
            epochs=total_epochs_trained + epochs_per_increment,
            validation_data=(X_val_fold, y_val_fold),
            callbacks=[early_stopping, tensorboard_callback],
            verbose=1
        )

        # Evaluate the model
        val_loss, val_mae = model.evaluate(X_val_fold, y_val_fold, verbose=0)
        print(f"Fold {fold}, Total Epochs Trained: {total_epochs_trained + epochs_per_increment}, Val MAE: {val_mae}")

        # Check for improvement
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            model.save(model_path)
            no_improvement_counter = 0
        else:
            no_improvement_counter += 1

        total_epochs_trained += epochs_per_increment

        if no_improvement_counter >= 2:
            print(f"No improvement in validation MAE for {no_improvement_counter} increments. Stopping training for fold {fold}.")
            break

    return {
        'fold': fold,
        'loss': val_loss,
        'mae': best_val_mae,
        'best_model': model
    }

if __name__ == '__main__':
    # Prepare data for each fold
    fold_data_list = [
        (fold, train_index, val_index)
        for fold, (train_index, val_index) in enumerate(kf.split(X_selected), 1)
    ]

    # Initialize lists to store results
    all_losses = []
    all_maes = []
    best_models = []

    # Run folds sequentially instead of concurrently
    for fold_data in fold_data_list:
        fold = fold_data[0]
        model_path = f'insurance_best_model_fold_{fold}.h5'
        if os.path.exists(model_path):
            print(f"Model for fold {fold} already exists. Skipping.")
            continue
        if fold == 4:
            print(f"Completing existing trials for fold {fold}.")
        elif fold == 5:
            print(f"Proceeding with new plan for fold {fold}.")
        else:
            print(f"Proceeding with existing plan for fold {fold}.")
        result = train_fold(fold_data)
        if result['loss'] is not None:
            all_losses.append(result['loss'])
            all_maes.append(result['mae'])
            best_models.append(result['best_model'])
            fold = result['fold']
            print(f"Fold {fold} completed.")
        else:
            fold = result['fold']
            print(f"Fold {fold} did not complete successfully.")

    # ...existing code to select the best overall model...

    # After all folds, select the overall best model if available
    if best_models:
        best_fold = np.argmin(all_maes) + 1
        best_overall_model = best_models[np.argmin(all_maes)]
        overall_model_path = 'insurance_best_overall_model.h5'
        best_overall_model.save(overall_model_path)
        print(f"Best overall model from fold {best_fold} saved as {overall_model_path}")
        logging.info(f"Best overall model from fold {best_fold} saved as {overall_model_path}")
    else:
        print("No models were successfully trained.")
        logging.warning("No models were successfully trained.")

    # Print average metrics
    if all_losses and all_maes:
        avg_loss = np.mean(all_losses)
        avg_mae = np.mean(all_maes)
        print(f"Average Validation Loss: {avg_loss}")
        print(f"Average Validation MAE: {avg_mae}")
        logging.info(f"Average Validation Loss: {avg_loss}")
        logging.info(f"Average Validation MAE: {avg_mae}")

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import joblib

# Load the test dataset
test_data = pd.read_csv('cleaned_test.csv')
print("Test data loaded from 'cleaned_test.csv'")

# Save 'id' column for submission
ids = test_data['id']

# Drop 'PremiumAmount' column if it exists
if 'PremiumAmount' in test_data.columns:
    test_data = test_data.drop('PremiumAmount', axis=1)

# Load preprocessing objects
feature_names = joblib.load('feature_names.joblib')
print("Feature names loaded.")
scaler = joblib.load('scaler.joblib')
print("Scaler loaded.")
selector = joblib.load('selector.joblib')
print("Selector loaded.")
cleaned_categorical_columns = joblib.load('cleaned_categorical_columns.joblib')
print("Categorical columns loaded.")

# Preprocess test data
test_data.fillna(0, inplace=True)

# Apply encoding
test_categorical_columns = [col for col in cleaned_categorical_columns if col in test_data.columns]
test_data_encoded = pd.get_dummies(test_data, columns=test_categorical_columns, drop_first=True)

# Reindex to match training features
test_data_encoded = test_data_encoded.reindex(columns=feature_names, fill_value=0)

# Apply scaling
X_test_scaled = scaler.transform(test_data_encoded)

# Apply feature selection
X_test_selected = selector.transform(X_test_scaled)

# Load the trained model (use one of the saved models)
model = load_model('insurance_best_model_fold_1.h5')
print("Model loaded.")

# Make predictions on the test data
predictions = model.predict(X_test_selected)
predictions = np.expm1(predictions).flatten()  # Reverse log transformation

# Create a DataFrame for the submission
submission = pd.DataFrame({'id': ids, 'Premium Amount': predictions})

# Save the submission file
submission.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'")
